# RS HW2 · Q1 — BPR-MF (Z-score / Temporal Validation)

**Reference implementation:** [gamboviol/bpr](https://github.com/gamboviol/bpr). This notebook reimplements the BPR pairwise ranking objective in PyTorch and adds item bias, user-wise rating Z-score, temporal validation, full-catalog NDCG@20, and Kaggle submission.

**Evaluation assumptions (not confirmed competition rules):** validation relevant movie = `rating >= 4.0`; full-catalog ranking, previously observed training movies excluded; average NDCG over target users with at least one relevant validation item. Keep this evaluation protocol fixed while tuning training thresholds. The Kaggle leaderboard may use another ground-truth definition.

Run in Kaggle with the competition data attached, or locally with the original Kaggle ZIP in the notebook folder.

In [35]:
from pathlib import Path
import zipfile, random, copy
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

COMPETITION_DIR = Path('/kaggle/input/competitions/skku-esw-4024-recommender-system')
LOCAL_DIR = Path('data')
ZIP_PATH = Path('skku-esw-4024-recommender-system.zip')
if COMPETITION_DIR.joinpath('train.csv').exists():
    DATA_DIR = COMPETITION_DIR
else:
    DATA_DIR = LOCAL_DIR
    if not DATA_DIR.joinpath('train.csv').exists():
        if not ZIP_PATH.exists() and Path('/mnt/data/skku-esw-4024-recommender-system.zip').exists():
            ZIP_PATH = Path('/mnt/data/skku-esw-4024-recommender-system.zip')
        if not ZIP_PATH.exists():
            raise FileNotFoundError('Attach Kaggle competition data or put the original ZIP beside notebook.')
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(ZIP_PATH) as z:
            for filename in ('train.csv', 'test.csv', 'sample_submission.csv'):
                (DATA_DIR / filename).write_bytes(z.read(filename))

OUTPUT_DIR = Path('/kaggle/working/outputs') if Path('/kaggle/working').exists() else Path('outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_PATH, TEST_PATH, SAMPLE_PATH = [DATA_DIR / f for f in ('train.csv', 'test.csv', 'sample_submission.csv')]

# Shared evaluation settings
K = 20
VAL_RATIO = 0.20
VAL_RELEVANT_RATING = 4.0  # fixed across experiments, not verified Kaggle rule

# BPR training settings: adjust these for experiments
Z_THRESHOLD = 0.0
LATENT_DIM = 64
USE_ITEM_BIAS = True
LR = 0.003
BATCH_SIZE = 1024
EPOCHS = 100
PATIENCE = 10
L2_REG = 1e-4
NEG_SAMPLING = 'popularity_mixed'  # 'uniform' or 'popularity_mixed'
POPULARITY_ALPHA = 0.5
POPULARITY_BETA = 0.75

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Data:', DATA_DIR, '| Device:', DEVICE, '| Output:', OUTPUT_DIR)

Data: /kaggle/input/competitions/skku-esw-4024-recommender-system | Device: cpu | Output: /kaggle/working/outputs


## 1. Load and inspect

In [36]:
train_raw = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample = pd.read_csv(SAMPLE_PATH)
assert {'userId','movieId','rating','time_idx'} <= set(train_raw.columns)
assert {'id','userId'} <= set(test.columns)
assert {'id','recommendations'} <= set(sample.columns)
assert not train_raw.duplicated(['userId','movieId']).any(), 'Duplicate user–movie interactions exist; decide aggregation first.'
assert set(test.userId).issubset(set(train_raw.userId)), 'Unexpected test cold-start users.'
assert sample['id'].tolist() == test['id'].tolist(), 'Submission IDs do not match test order.'
print('Train:', train_raw.shape, '| Test:', test.shape)
print('Users:', train_raw.userId.nunique(), '| Movies:', train_raw.movieId.nunique())
print('Ratings:'); print(train_raw.rating.value_counts().sort_index())
display(train_raw.head(),test.head(),sample.head(2))

Train: (80621, 4) | Test: (588, 2)
Users: 610 | Movies: 8237
Ratings:
rating
0.5      999
1.0     2170
1.5     1353
2.0     5994
2.5     4455
3.0    16235
3.5    10224
4.0    21640
4.5     6712
5.0    10839
Name: count, dtype: int64


,userId,movieId,rating,time_idx
0,0,3086,3.0,0
1,0,5849,2.0,1
2,0,3134,3.0,2
3,0,1948,5.0,3
4,0,383,4.0,4


,id,userId
0,0,0
1,1,1
2,2,2
3,3,3
4,4,4


,id,recommendations
0,0,4830 2814 6872 1040 6398 5783 9472 7593 5677 3...
1,1,2573 7753 4324 8635 7256 5889 9597 4762 5233 2...


## 2. Temporal 80:20 hold-out

For each user, keep the earliest ~80% of **all observed ratings** for training and the last ~20% for validation. Users need at least one training record. Validation relevant movies are fixed as `rating >= VAL_RELEVANT_RATING`; for mean NDCG we evaluate target users with at least one such movie. We intentionally do **not** calculate user statistics using validation ratings.

In [37]:
def temporal_split(df, val_ratio=0.2):
    ordered = df.sort_values(['userId','time_idx'], kind='stable')
    groups = ordered.groupby('userId', sort=False)
    counts = groups['userId'].transform('size')
    positions = groups.cumcount()
    cutoffs = np.maximum(1, np.floor(counts * (1-val_ratio)).astype(int))
    tr = ordered.loc[positions < cutoffs].copy().reset_index(drop=True)
    va = ordered.loc[positions >= cutoffs].copy().reset_index(drop=True)
    return tr, va

train_part, val_part = temporal_split(train_raw, VAL_RATIO)
val_relevant = val_part.loc[
    val_part['rating'].ge(VAL_RELEVANT_RATING) & val_part['userId'].isin(test.userId),
    ['userId','movieId']
].copy()
print(f'Train interactions: {len(train_part):,} | Validation interactions: {len(val_part):,}')
print(f'Validation relevant interactions: {len(val_relevant):,} | Evaluated users: {val_relevant.userId.nunique():,}')
assert not set(zip(train_part.userId,train_part.movieId)) & set(zip(val_part.userId,val_part.movieId))

Train interactions: 64,246 | Validation interactions: 16,375
Validation relevant interactions: 7,400 | Evaluated users: 575


## 3. Global ID mapping; Z-score fitted on training data only

In [38]:
user_ids = pd.Index(pd.unique(pd.concat([train_raw.userId,test.userId],ignore_index=True)))
item_ids = pd.Index(pd.unique(train_raw.movieId))
user2idx = {int(uid): i for i,uid in enumerate(user_ids)}
item2idx = {int(iid): i for i,iid in enumerate(item_ids)}
idx2item = item_ids.to_numpy()
num_users, num_items = len(user_ids), len(item_ids)

def encode_frame(df):
    out = df.copy()
    out['u'] = out.userId.map(user2idx)
    out['i'] = out.movieId.map(item2idx)
    if out[['u','i']].isna().any().any():
        raise ValueError('Unknown user / movie ID (global ID map only contains train_raw movies).')
    out[['u','i']] = out[['u','i']].astype(np.int64)
    return out

def make_histories(train_df, z_threshold):
    # Critically: fit mean/std exclusively on the dataframe passed in.
    frame = encode_frame(train_df)
    by_user = frame.groupby('userId')['rating']
    avg = by_user.transform('mean')
    sd = by_user.transform('std').fillna(0.0)
    frame['rating_z'] = ((frame.rating-avg)/sd.replace(0.0,np.nan)).fillna(0.0)
    positives = frame.loc[frame.rating_z >= z_threshold].copy()
    observed = {int(u): set(g.i.to_numpy().tolist()) for u,g in frame.groupby('u')}
    positive_pairs = positives[['u','i']].to_numpy(dtype=np.int64)
    pop = np.bincount(frame['i'].to_numpy(),minlength=num_items).astype(np.float64)
    return observed, positive_pairs, pop, frame

observed_train, pairs_train, item_pop_train, train_encoded = make_histories(train_part,Z_THRESHOLD)
val_encoded = encode_frame(val_relevant)
truth = {int(u): set(g.i.to_numpy().tolist()) for u,g in val_encoded.groupby('u')}
print('Positive train interactions:',len(pairs_train),'| users with positives:',len(set(pairs_train[:,0].tolist())) if len(pairs_train) else 0)
print('Training rating_z distribution:'); print(train_encoded.rating_z.describe())
assert len(pairs_train)>0, 'No positive pairs: lower Z_THRESHOLD.' 

Positive train interactions: 35134 | users with positives: 610
Training rating_z distribution:
count    6.424600e+04
mean     2.322541e-18
std      9.950457e-01
min     -5.777821e+00
25%     -5.651389e-01
50%      1.235517e-01
75%      6.986419e-01
max      4.248529e+00
Name: rating_z, dtype: float64


## 4. BPR-MF with item bias

Score: `user embedding · item embedding + item bias`. The user bias would cancel out in BPR pairwise differences. Negatives are sampled from *unobserved training items*, not low-rated observed movies. `L2_REG` applies to the minibatch user, positive item, negative item embeddings (and used item biases).

In [39]:
class BPRMF(nn.Module):
    def __init__(self, n_users, n_items, dim, use_item_bias=True):
        super().__init__()
        self.user_emb = nn.Embedding(n_users,dim)
        self.item_emb = nn.Embedding(n_items,dim)
        self.use_item_bias = use_item_bias
        self.item_bias = nn.Embedding(n_items,1) if use_item_bias else None
        nn.init.normal_(self.user_emb.weight,std=0.05)
        nn.init.normal_(self.item_emb.weight,std=0.05)
        if self.item_bias is not None: nn.init.zeros_(self.item_bias.weight)

    def forward(self,u,i):
        scores = (self.user_emb(u)*self.item_emb(i)).sum(-1)
        if self.item_bias is not None:
            scores = scores + self.item_bias(i).flatten()
        return scores

    def scores_for_users(self,u):
        scores = self.user_emb(u) @ self.item_emb.weight.T
        if self.item_bias is not None:
            scores = scores + self.item_bias.weight.flatten().unsqueeze(0)
        return scores

    def batch_l2(self,u,i,j):
        reg = (self.user_emb(u).pow(2).sum(-1)+
               self.item_emb(i).pow(2).sum(-1)+
               self.item_emb(j).pow(2).sum(-1))
        if self.item_bias is not None:
            reg = reg+self.item_bias(i).flatten().pow(2)+self.item_bias(j).flatten().pow(2)
        return reg.mean()

def negative_probs(pop, sampling='uniform', alpha=0.5, beta=0.75):
    uniform = np.full(len(pop),1.0/len(pop))
    if sampling=='uniform':return uniform
    if sampling!='popularity_mixed':raise ValueError('Unsupported NEG_SAMPLING')
    popularity = np.power(pop+1e-6,beta)
    popularity /= popularity.sum()
    return (1-alpha)*uniform + alpha*popularity

def sample_negatives(users, observed, probs, rng):
    # Rejection sampling: never sample a training-observed movie for that user.
    sampled = rng.choice(len(probs),size=len(users),p=probs)
    bad = np.fromiter((int(i) in observed.get(int(u),set()) for u,i in zip(users,sampled)),bool,count=len(users))
    while bad.any():
        sampled[bad] = rng.choice(len(probs),size=int(bad.sum()),p=probs)
        bad = np.fromiter((int(i) in observed.get(int(u),set()) for u,i in zip(users,sampled)),bool,count=len(users))
    return sampled

def train_epoch(model,optimizer,pairs,observed,probs,seed):
    model.train()
    rng = np.random.default_rng(seed)
    perm = rng.permutation(len(pairs))
    total_loss=0.0
    for inds in np.array_split(perm, max(1,int(np.ceil(len(pairs)/BATCH_SIZE)))):
        ui = pairs[inds]
        uu,ii = ui[:,0],ui[:,1]
        jj = sample_negatives(uu, observed, probs, rng)
        u = torch.as_tensor(uu,device=DEVICE)
        i = torch.as_tensor(ii,device=DEVICE)
        j = torch.as_tensor(jj,device=DEVICE)
        loss = F.softplus(-(model(u,i)-model(u,j))).mean() + L2_REG*model.batch_l2(u,i,j)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        total_loss += float(loss.item())*len(inds)
    return total_loss/len(pairs)

probs_train = negative_probs(item_pop_train, NEG_SAMPLING,POPULARITY_ALPHA,POPULARITY_BETA)

## 5. Full-catalog Top-20 NDCG

Exclude all movies observed in the **visible training split**, including low-rated ones. Validation movies remain eligible. Evaluate only users with at least one `rating >= 4` holdout movie; print that count so the scope is transparent.

In [40]:
@torch.no_grad()
def recommend_idx(model, user_indices, observed, k=20, batch_users=128, popularity=None):
    model.eval()
    result=[]
    for start in range(0,len(user_indices),batch_users):
        users = np.asarray(user_indices[start:start+batch_users],dtype=np.int64)
        score = model.scores_for_users(torch.as_tensor(users,device=DEVICE)).cpu().numpy()
        for r,u in enumerate(users):
            seen = observed.get(int(u),set())
            if seen: score[r,list(seen)] = -np.inf
            # A user without train-positive interactions has no trained user embedding.
            # Use observed train-item popularity for a deterministic cold-positive fallback.
            if popularity is not None and int(u) not in trained_users:
                score[r] = popularity
                if seen:score[r,list(seen)] = -np.inf
            top = np.argpartition(-score[r],kth=k-1)[:k]
            top = top[np.argsort(-score[r,top],kind='stable')]
            result.append(top)
    return np.asarray(result,dtype=np.int64)

def ndcg_at_k(model,truth,observed,popularity,trained_users_set,k=20):
    global trained_users
    trained_users = trained_users_set
    users = np.asarray(sorted(truth.keys()),dtype=np.int64)
    if not len(users):raise ValueError('No users with validation relevant movies')
    recommendations = recommend_idx(model,users,observed,k,popularity=popularity)
    vals=[];hits=[]
    for rec,u in zip(recommendations,users):
        gt = truth[int(u)]
        gain = sum(1/np.log2(rank+2) for rank,item in enumerate(rec) if int(item) in gt)
        ideal = sum(1/np.log2(rank+2) for rank in range(min(k,len(gt))))
        vals.append(gain/ideal)
        hits.append(any(int(item) in gt for item in rec))
    return float(np.mean(vals)),float(np.mean(hits))

# Fallback is based only on visible training interactions.
trained_users_train = set(int(x) for x in pairs_train[:,0])
print('Validation evaluation users:',len(truth))

Validation evaluation users: 575


## 6. Train with early stopping; retain best model

In [41]:
torch.manual_seed(SEED)
model = BPRMF(num_users,num_items,LATENT_DIM,USE_ITEM_BIAS).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(),lr=LR)
best_ndcg=-1.0; best_epoch=0; best_state=None; wait=0; history=[]
for epoch in range(1,EPOCHS+1):
    train_loss = train_epoch(model,optimizer,pairs_train,observed_train,probs_train,SEED+epoch)
    val_ndcg,val_hit = ndcg_at_k(model,truth,observed_train,item_pop_train,trained_users_train,K)
    history.append({'epoch':epoch,'train_loss':train_loss,'val_ndcg20':val_ndcg,'val_hit20':val_hit})
    print(f'Epoch {epoch:3d} | loss {train_loss:.5f} | NDCG@20 {val_ndcg:.5f} | Hit@20 {val_hit:.5f}')
    if val_ndcg > best_ndcg + 1e-7:
        best_ndcg=val_ndcg; best_epoch=epoch
        best_state=copy.deepcopy(model.state_dict());wait=0
    else:
        wait+=1
        if wait>=PATIENCE:
            print('Early stopping triggered.');break
model.load_state_dict(best_state)
history_df = pd.DataFrame(history)
display(history_df.sort_values('val_ndcg20',ascending=False).head(10))
print(f'Best: Epoch={best_epoch}, NDCG@20={best_ndcg:.6f}')
history_df.to_csv(OUTPUT_DIR/'q1_bpr_v2_history.csv',index=False)

Epoch   1 | loss 0.67952 | NDCG@20 0.04493 | Hit@20 0.34261
Epoch   2 | loss 0.63074 | NDCG@20 0.07810 | Hit@20 0.44696
Epoch   3 | loss 0.52064 | NDCG@20 0.09880 | Hit@20 0.47478
Epoch   4 | loss 0.34324 | NDCG@20 0.10296 | Hit@20 0.46609
Epoch   5 | loss 0.23774 | NDCG@20 0.10444 | Hit@20 0.47478
Epoch   6 | loss 0.19283 | NDCG@20 0.10336 | Hit@20 0.48000
Epoch   7 | loss 0.16299 | NDCG@20 0.10321 | Hit@20 0.47478
Epoch   8 | loss 0.13926 | NDCG@20 0.10412 | Hit@20 0.48870
Epoch   9 | loss 0.12430 | NDCG@20 0.10626 | Hit@20 0.50087
Epoch  10 | loss 0.10761 | NDCG@20 0.10470 | Hit@20 0.49565
Epoch  11 | loss 0.09571 | NDCG@20 0.10520 | Hit@20 0.49739
Epoch  12 | loss 0.08630 | NDCG@20 0.10682 | Hit@20 0.49043
Epoch  13 | loss 0.07765 | NDCG@20 0.10683 | Hit@20 0.50261
Epoch  14 | loss 0.07209 | NDCG@20 0.10950 | Hit@20 0.50957
Epoch  15 | loss 0.06539 | NDCG@20 0.11057 | Hit@20 0.52174
Epoch  16 | loss 0.06158 | NDCG@20 0.11047 | Hit@20 0.51652
Epoch  17 | loss 0.05662 | NDCG@20 0.111

,epoch,train_loss,val_ndcg20,val_hit20
39,40,0.021845,0.123910,0.547826
38,39,0.022615,0.123802,0.558261
37,38,0.022744,0.123466,0.546087
40,41,0.020552,0.122362,0.544348
41,42,0.021370,0.122124,0.561739
36,37,0.024140,0.121317,0.560000
33,34,0.025713,0.120946,0.566957
32,33,0.026618,0.119197,0.556522
35,36,0.024795,0.118738,0.544348
43,44,0.021339,0.118448,0.561739


Best: Epoch=40, NDCG@20=0.123910


## 7. Refit on ALL of train.csv

Recalculate each user's Z-score from the entire training dataset and train a new model for `best_epoch` epochs. Epoch count is selected by validation; this is a pragmatic approach, not a guarantee that the refit optimum has exactly the same number of epochs.

In [42]:
observed_full,pairs_full,item_pop_full,full_encoded = make_histories(train_raw,Z_THRESHOLD)
trained_users_full = set(int(x) for x in pairs_full[:,0])
probs_full = negative_probs(item_pop_full,NEG_SAMPLING,POPULARITY_ALPHA,POPULARITY_BETA)
print('Full training positives:',len(pairs_full),'| Trained users:',len(trained_users_full))
torch.manual_seed(SEED)
final_model = BPRMF(num_users,num_items,LATENT_DIM,USE_ITEM_BIAS).to(DEVICE)
final_optimizer = torch.optim.Adam(final_model.parameters(),lr=LR)
for epoch in range(1,best_epoch+1):
    final_loss = train_epoch(final_model,final_optimizer,pairs_full,observed_full,probs_full,SEED+epoch)
    print(f'Final refit {epoch:3d}/{best_epoch}: loss {final_loss:.5f}')

Full training positives: 43772 | Trained users: 610
Final refit   1/40: loss 0.67662
Final refit   2/40: loss 0.61303
Final refit   3/40: loss 0.44697
Final refit   4/40: loss 0.28299
Final refit   5/40: loss 0.21676
Final refit   6/40: loss 0.18063
Final refit   7/40: loss 0.15650
Final refit   8/40: loss 0.13111
Final refit   9/40: loss 0.11612
Final refit  10/40: loss 0.10047
Final refit  11/40: loss 0.08830
Final refit  12/40: loss 0.08167
Final refit  13/40: loss 0.07129
Final refit  14/40: loss 0.06579
Final refit  15/40: loss 0.06147
Final refit  16/40: loss 0.05635
Final refit  17/40: loss 0.05254
Final refit  18/40: loss 0.04914
Final refit  19/40: loss 0.04573
Final refit  20/40: loss 0.04313
Final refit  21/40: loss 0.04148
Final refit  22/40: loss 0.03760
Final refit  23/40: loss 0.03742
Final refit  24/40: loss 0.03532
Final refit  25/40: loss 0.03379
Final refit  26/40: loss 0.03288
Final refit  27/40: loss 0.03164
Final refit  28/40: loss 0.03090
Final refit  29/40: loss

## 8. Create and validate the Kaggle submission

In [43]:
trained_users = trained_users_full
encoded_test_users = np.asarray([user2idx[int(u)] for u in test.userId],dtype=np.int64)
recs_encoded = recommend_idx(final_model,encoded_test_users,observed_full,k=K,popularity=item_pop_full)
recs_original = idx2item[recs_encoded]
submission = pd.DataFrame({
    'id':test['id'].to_numpy(),
    'recommendations':[' '.join(map(str, row)) for row in recs_original]
})
assert list(submission.columns)==list(sample.columns)
assert submission['id'].tolist()==sample['id'].tolist()
assert len(submission)==len(sample)
assert all(len(v.split())==K and len(set(v.split()))==K for v in submission.recommendations)
for uid,rec in zip(encoded_test_users,recs_encoded):
    assert not set(rec) & observed_full.get(int(uid),set())
output_file=OUTPUT_DIR/'submission_q1_bpr_v2_exp4.csv'
submission.to_csv(output_file,index=False)
print('Saved:',output_file)
display(submission.head())

Saved: /kaggle/working/outputs/submission_q1_bpr_v2_exp4.csv


,id,recommendations
0,0,4480 2344 7093 2349 3528 2723 6298 3949 2982 6...
1,1,6089 1555 3261 7304 5650 7837 6958 3878 6607 9...
2,2,2344 6281 1008 3527 7332 1555 5361 7263 5466 5...
3,3,6281 8819 6685 3713 7888 3727 7332 290 9561 69...
4,4,6281 4115 2256 3727 1037 3011 5842 8112 17 84 ...


## 9. Suggested experiment log

Keep the exact **same temporal split and fixed validation relevance definition** to compare changes fairly. Suggested experiments: `Z_THRESHOLD` 0 vs 0.5; `USE_ITEM_BIAS` False vs True; `NEG_SAMPLING` uniform vs popularity_mixed; `LATENT_DIM` 32/64/128; `LR` 0.001/0.003; `L2_REG` 1e-5/1e-4. Re-run from the first configuration cell for each experiment. Do **not** use Kaggle public leaderboard as the only hyperparameter-selection signal. Submit CSV, record actual public score and screenshot for HW2 report.